# Windrose Tilt Summaries

Direction and magnitude summaries for AE/CE tilt vectors.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import seacofs_tilt_tools as tilt

paths = tilt.Paths()
grid = tilt.load_grid(paths.grid, paths.z_r)
df_eddies, df_tilt = tilt.load_tilt_tables(paths)
mag_bins = [0, 10, 20, 30, 40, np.inf]
df_eddies.head()


In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(9, 4.5), subplot_kw={"projection": "polar"}, constrained_layout=True)
for ax, cyc, colors in zip(axs, ["AE", "CE"], [plt.cm.Reds(np.linspace(0.15, 1, len(mag_bins) - 1)), plt.cm.Blues(np.linspace(0.15, 1, len(mag_bins) - 1))]):
    subset = df_eddies.loc[df_eddies.Cyc == cyc].dropna(subset=["TiltDir", "TiltDis"])
    tilt.plot_windrose(ax, subset, title=cyc, mag_bins=mag_bins, colors=colors)
axs[0].legend(title="Tilt dist. (km)", loc="upper left", bbox_to_anchor=(1.05, 1.05), frameon=False)
plt.show()


In [ ]:
df_regions, bin_grid = tilt.assign_six_regions(df_eddies, grid)
fig, axs = plt.subplots(2, 6, figsize=(15, 6), subplot_kw={"projection": "polar"}, constrained_layout=True)

for col, bin_id in enumerate(range(1, 7)):
    for row, cyc in enumerate(["AE", "CE"]):
        ax = axs[row, col]
        subset = df_regions.loc[(df_regions.bin_id == bin_id) & (df_regions.Cyc == cyc)].dropna(subset=["TiltDir", "TiltDis"])
        colors = plt.cm.Reds(np.linspace(0.15, 1, len(mag_bins) - 1)) if cyc == "AE" else plt.cm.Blues(np.linspace(0.15, 1, len(mag_bins) - 1))
        if len(subset):
            tilt.plot_windrose(ax, subset, title=f"{cyc} bin {bin_id}", mag_bins=mag_bins, colors=colors)
        else:
            ax.set_axis_off()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(5, 5), constrained_layout=True)
ax.contourf(grid.X_grid, grid.Y_grid, np.where(grid.mask_rho == 0, 1, np.nan), levels=[0.5, 1.5], colors=["k"], alpha=0.5)
ax.contourf(grid.X_grid, grid.Y_grid, bin_grid, levels=np.arange(0.5, 7.5, 1), alpha=0.25, cmap="gist_rainbow")
ax.contour(grid.X_grid, grid.Y_grid, grid.h, levels=[4000], colors="k", linewidths=1)
for x, y, label in [(220, 1300, "S1"), (120, 50, "S2"), (400, 1450, "U1"), (800, 1450, "U2"), (400, 700, "D1"), (800, 700, "D2")]:
    ax.text(x, y, label, ha="center", va="center", fontweight="bold")
ax.set_aspect("equal")
ax.set_xlabel("x (km)")
ax.set_ylabel("y (km)")
plt.show()
